## Imports:

In [ ]:
import pandas as pd
import optuna
import torch
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
import numpy as np
from lightgbm import early_stopping
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from scipy.optimize import minimize
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import OrdinalEncoder


## Data loading:

In [ ]:
train_data = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

## Data preprocessing:

In [ ]:
train_data.head()

In [ ]:
train_data.isna().sum()

In [ ]:
# NUMERICAL DATA

# 1. DEFINITIONS 
def gauss(x):
    return np.exp(-(x - 8) ** 2 / 4.5)

def engineer_features(df):
    res = df.copy()
    health_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level']
    existing_health_cols = [c for c in health_cols if c in res.columns]
    if existing_health_cols:
        res['healthy_score'] = res[existing_health_cols].mean(axis=1)
    
    if 'sleep_duration' in res.columns and 'sleep_quality' in res.columns:
        sleep_score_map = {0: 0.5, 1: 1.0, 2: 1.5}
        res['sleep_score'] = gauss(res['sleep_duration']) * res['sleep_quality'].map(sleep_score_map)
        
    if 'calorie_expenditure' in res.columns and 'bmi' in res.columns:
        res['calorie_score'] = res['calorie_expenditure'] / res['bmi']
        
    if 'step_count' in res.columns and 'bmi' in res.columns:
        res['steps_score'] = res['step_count'] / res['bmi']
        
    if 'step_count' in res.columns and 'exercise_duration' in res.columns and 'physical_activity_level' in res.columns:
        activity_map = {0: 0.5, 1: 1.0, 2: 1.5}
        res['sport_score'] = np.log1p(res['step_count'] * res['exercise_duration'] * res['physical_activity_level'].map(activity_map))
        
    if 'heart_rate' in res.columns:
        res['bradycardia'] = np.where(res['heart_rate'].isna(), np.nan, (res['heart_rate'] < 60).astype(int))
        res['tachycardia'] = np.where(res['heart_rate'].isna(), np.nan, (res['heart_rate'] > 100).astype(int))
        
    if 'bmi' in res.columns:
        res['underweight'] = np.where(res['bmi'].isna(), np.nan, (res['bmi'] < 18.5).astype(int))
        res['overweight'] = np.where(res['bmi'].isna(), np.nan, (res['bmi'] > 25).astype(int))
        
    return res

# 2. NUMERICAL COLUMNS & HEATMAP FEATURES
base_num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

for df in [train_data, test_data]:
    df['calories_per_step'] = df['calorie_expenditure'] / (df['step_count'] + 1)
    df['calories_per_minute'] = df['calorie_expenditure'] / (df['exercise_duration'] + 1)
    df['steps_per_minute'] = df['step_count'] / (df['exercise_duration'] + 1)
    df['total_activity_score'] = df['step_count'] * df['exercise_duration']

num_cols = base_num_cols + ['calories_per_step', 'calories_per_minute', 'steps_per_minute', 'total_activity_score']

# 3. MISSING VALUES INDICATORS
for col in num_cols:
    train_data[f'{col}_was_missing'] = train_data[col].isnull().astype(int)
    test_data[f'{col}_was_missing'] = test_data[col].isnull().astype(int)

train_medians = train_data[num_cols].median()

# 4. BASIC CATEGORICAL BINNING
for df in [train_data, test_data]:
    if 'calorie_expenditure' in df.columns:
        df['calorie_cat'] = (df['calorie_expenditure'] // 5).fillna(-1).astype(str)
    if 'water_intake' in df.columns:
        df['water_cat'] = (df['water_intake'] * 50).fillna(-1).astype(int).astype(str)
    if 'step_count' in df.columns:
        df['step_count_rounded'] = df['step_count'].round(-1).fillna(-1).astype(str)

# 5. QUANTILE BINS LOGIC 
bin_config = {'sleep_duration':[70], 'water_intake':[10]}
category_map_bins = {}

for col, bins_list in bin_config.items():
    if col in train_data.columns:
        for n_bins in bins_list:
            bin_name = f'{col}_{n_bins}_quantile_bin_'
            kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
            train_data[bin_name] = kb.fit_transform(train_data[[col]].fillna(train_medians[col])).ravel().astype('int32').astype(str)
            category_map_bins[bin_name] = kb
            if col in test_data.columns:
                test_data[bin_name] = category_map_bins[bin_name].transform(test_data[[col]].fillna(train_medians[col])).ravel().astype('int32').astype(str)

# 6. FILLING NUMERICAL MISSING VALUES
train_data[num_cols] = train_data[num_cols].fillna(train_medians)
test_data[num_cols] = test_data[num_cols].fillna(train_medians)


In [ ]:
# CATEGORICAL DATA

cat_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

new_features = ['calorie_cat', 'water_cat', 'step_count_rounded', 'sleep_duration_70_quantile_bin_', 'water_intake_10_quantile_bin_']
cat_cols.extend(new_features)

train_data[cat_cols] = train_data[cat_cols].fillna('Missing')
test_data[cat_cols] = test_data[cat_cols].fillna('Missing')


print(f"[INFO] Total categorical features encoded: {len(cat_cols)}")

In [ ]:
# ENCODING (for XGBoost & LGBM)

X_train_cat = train_data.copy()
X_test_cat = test_data.copy()

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_data[cat_cols] = encoder.fit_transform(train_data[cat_cols].astype(str))
test_data[cat_cols] = encoder.transform(test_data[cat_cols].astype(str))

train_data = engineer_features(train_data)
test_data = engineer_features(test_data)

In [ ]:
train_data.isna().sum()

In [ ]:
train_data.head()

In [ ]:
# FEATURES AND LABEL CREATING

# ---------------------------------------------------------
# y_train creating
# ---------------------------------------------------------
y_train_raw = train_data['health_condition']
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)

# ---------------------------------------------------------
# X_train and X_test creating
# ---------------------------------------------------------
X_train = train_data.drop(columns=['id', 'health_condition'])
X_test = test_data.drop(columns=['id'])

## Models creating:

In [ ]:
# ENSEMBLE

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_test_total = np.zeros((len(X_test), 3))
cat_test_total = np.zeros((len(X_test), 3))
xgb_test_total = np.zeros((len(X_test), 3))

print("Training is starting...")

for fold, (train_idxs, val_idxs) in enumerate(kf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idxs], y_train[train_idxs]
    X_va, y_va = X_train.iloc[val_idxs], y_train[val_idxs]

# ---------------------------------------------------------
# LightGBM
# ---------------------------------------------------------
    lgb_model = LGBMClassifier(
        n_estimators=1500, 
        learning_rate=0.05, 
        max_depth=6, 
        num_leaves=31, 
        class_weight='balanced', 
        subsample=0.8, 
        colsample_bytree=0.8, 
        random_state=42, 
        n_jobs=-1, 
        verbose=-1)
    
    lgb_model.fit(
        X_tr, 
        y_tr, 
        eval_set=[(X_va, y_va)], 
        callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    lgb_test_total += lgb_model.predict_proba(X_test) / kf.n_splits

# ---------------------------------------------------------
# CatBoost 
# ---------------------------------------------------------
    cat_model = CatBoostClassifier(
        iterations=1500, 
        learning_rate=0.075, 
        depth=8, 
        l2_leaf_reg=7.0, 
        random_seed=42, 
        subsample=0.85,
        bootstrap_type="Bernoulli",
        early_stopping_rounds=50, 
        verbose=0
    )
    
    X_tr_cat = X_tr.copy()
    X_tr_cat[cat_cols] = X_train_cat[cat_cols].iloc[train_idxs].astype(str)
    
    X_va_cat = X_va.copy()
    X_va_cat[cat_cols] = X_train_cat[cat_cols].iloc[val_idxs].astype(str)

    cat_model.fit(
        X_tr_cat, 
        y_tr, 
        eval_set=(X_va_cat, y_va),
        cat_features = cat_cols
    )
   
    
    cat_test_total += cat_model.predict_proba(X_test_cat) / kf.n_splits

# ---------------------------------------------------------
# XGBoost
# ---------------------------------------------------------

    sample_weight = compute_sample_weight(class_weight='balanced', y=y_tr)
    
    xgb_model = XGBClassifier(
        n_estimators=1500, 
        learning_rate=0.05, 
        max_depth=6, 
        subsample=0.8, 
        colsample_bytree=0.8, 
        random_state=42, 
        n_jobs=-1, 
        eval_metric='mlogloss', 
        early_stopping_rounds=50
    ) 
    
    xgb_model.fit(
        X_tr, 
        y_tr, 
        sample_weight=sample_weight, 
        eval_set=[(X_va, y_va)], 
        verbose=False
    )
    
    xgb_test_total += xgb_model.predict_proba(X_test) / kf.n_splits

    print(f"Fold {fold + 1}/5 succesfully completed!")

In [ ]:
# BLENDING

final_probs = ( 0.45 * cat_test_total + 0.45 * lgb_test_total + 0.10 * xgb_test_total ) 
final_classes = np.argmax(final_probs, axis=1) 
final_labels = label_encoder.inverse_transform(final_classes) 

In [ ]:
# CREATING SIBMISSION FILE

submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv') 
submission['health_condition'] = final_labels 
submission.to_csv('submission.csv', index=False) 
print("\nfile'submission.csv' is ready")